# Шаблон бэктеста QuantResearch

Шаблон Jupyter Notebook для исследования и запуска бэктестов в **Backtrader**. 
Запускать из окружения QuantResearch (`.venv`).

## 0. Результаты и заметки

На следующем этапе добавим метрики, сохранение результатов в SQLite и сравнение запусков. Пока можно записывать выводы эксперимента здесь.

**Гипотеза:** …

**Выводы:** …

In [2]:
#Загружает расширение autoreload
%load_ext autoreload
#Включает автоматическую перезагрузку импортированных модулей перед выполнением каждой ячейки
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Импорт библиотек

In [3]:
import sys
print(sys.executable)

/Volumes/DataD/AlgorithmicTrading/QuantResearch/.venv/bin/python


In [4]:
import backtrader as bt
import pandas as pd
import numpy as np

from IPython.display import display, HTML
from lightweight_charts import Chart
import matplotlib.pyplot as plt
# %matplotlib inlinefrom

In [ ]:
from FinamPy import FinamPy
from MOEXPy import MOEXPy
from FinLabPy.Core import Symbol

In [19]:
from IPython.display import display, HTML
# %matplotlib inline

## 3. Исторические данные

Backtrader ожидает DataFrame с индексом `DatetimeIndex` и столбцами `open`, `high`, `low`, `close`, `volume`.

In [6]:
df = pd.read_csv('../../Data/TQBR.CHMF_D1.txt', sep='\t')
df.head()

,datetime,open,high,low,close,volume
0,22.06.2005 00:00,220.0,220.0,217.0,219.0,64
1,23.06.2005 00:00,219.0,230.0,219.0,223.0,43
2,24.06.2005 00:00,225.0,227.0,216.0,216.0,18
3,27.06.2005 00:00,213.5,213.5,211.0,211.0,6
4,28.06.2005 00:00,214.0,216.0,208.0,208.0,43


In [7]:
df['datetime']  = pd.to_datetime(df['datetime'], format='%d.%m.%Y %H:%M').astype('datetime64[ns]')
df = df.set_index('datetime')
df.head()

,open,high,low,close,volume
datetime,,,,,
2005-06-22,220.0,220.0,217.0,219.0,64
2005-06-23,219.0,230.0,219.0,223.0,43
2005-06-24,225.0,227.0,216.0,216.0,18
2005-06-27,213.5,213.5,211.0,211.0,6
2005-06-28,214.0,216.0,208.0,208.0,43


In [8]:
print('Начало:', df.index.min())
print('Конец:', df.index.max())
print('Количество свечей:', len(df))

Начало: 2005-06-22 00:00:00
Конец: 2026-03-17 00:00:00
Количество свечей: 5267


In [21]:
import pandas as pd
from lightweight_charts import Chart

timeframe = 'D1'
title = 'CHMF'


chart = Chart()

# Тикер и таймфрейм в верхней панели
chart.topbar.textbox('symbol', title)
chart.topbar.textbox('timeframe', timeframe)

# Отображение OHLCV
chart.legend(visible=True)

# Для дневного таймфрейма скрываем часы
chart.time_scale(
    time_visible=False,
    seconds_visible=False
)

chart.set(df)
chart.show()

In [22]:
chart.exit()

## 5. Запуск бэктеста

In [34]:
# Создание движка
cerebro = bt.Cerebro()

# Добавление данных
cerebro.adddata(bt.feeds.PandasData(dataname=df))

#Начальный капитал
INITIAL_CASH = 1000_000.0 
cerebro.broker.setcash(INITIAL_CASH)

# Коммисия брокера Финам
from quant_tools.finam_commission import FinamCommission 
cerebro.broker.addcommissioninfo(FinamCommission())

# Размер позиции в процентах от капитала
cerebro.addsizer(
    bt.sizers.PercentSizerInt,
    percents=10,
)

In [35]:
# Добавление стратегии
from strategy import ResearchStrategy

FAST_PERIOD = 10
SLOW_PERIOD = 30

cerebro.addstrategy(
    ResearchStrategy,
    fast_period=FAST_PERIOD,
    slow_period=SLOW_PERIOD,
)


0

In [36]:
# 7. Анализаторы
cerebro.addanalyzer(
    bt.analyzers.TradeAnalyzer,
    _name="trades",
)

cerebro.addanalyzer(
    bt.analyzers.DrawDown,
    _name="drawdown",
)

cerebro.addanalyzer(
    bt.analyzers.SharpeRatio,
    _name="sharpe",
    timeframe=bt.TimeFrame.Days,
    annualize=True,
)

In [37]:

start_value = cerebro.broker.getvalue()
results = cerebro.run()
end_value = cerebro.broker.getvalue()

print(f"Стартовый капитал: {start_value:,.2f}")
print(f"Конечный капитал: {end_value:,.2f}")
# cerebro.plot(iplot=False)
# plt.show()

2005-09-20: EXECUTED BUY, price=260.00, size=393
2005-10-25: EXECUTED SELL, price=254.50, size=-393
2005-11-15: EXECUTED BUY, price=269.70, size=376
2006-04-21: EXECUTED SELL, price=368.00, size=-376
2006-06-08: EXECUTED BUY, price=330.00, size=312
2006-06-13: EXECUTED SELL, price=300.00, size=-312
2006-07-13: EXECUTED BUY, price=326.98, size=316
2006-09-05: EXECUTED SELL, price=330.00, size=-316
2006-10-11: EXECUTED BUY, price=339.50, size=302
2006-11-16: EXECUTED SELL, price=323.20, size=-302
2007-01-24: EXECUTED BUY, price=335.00, size=306
2007-03-13: EXECUTED SELL, price=336.03, size=-306
2007-03-28: EXECUTED BUY, price=351.50, size=290
2007-05-02: EXECUTED SELL, price=342.00, size=-290
2007-06-21: EXECUTED BUY, price=363.49, size=280
2007-08-15: EXECUTED SELL, price=398.00, size=-280
2007-09-05: EXECUTED BUY, price=438.02, size=236
2007-11-19: EXECUTED SELL, price=549.48, size=-236
2007-12-13: EXECUTED BUY, price=565.00, size=186
2008-01-25: EXECUTED SELL, price=532.25, size=-186


#### Вывод статистики анализаторов

In [17]:
strategy = results[0]

trades = strategy.analyzers.trades.get_analysis()
drawdown = strategy.analyzers.drawdown.get_analysis()
sharpe = strategy.analyzers.sharpe.get_analysis()

print("Закрытых сделок:", trades.total.get("closed", 0))
print("Прибыльных сделок:", trades.won.get("total", 0))
print("Убыточных сделок:", trades.lost.get("total", 0))
print(f"Максимальная просадка: {drawdown.max.drawdown:.2f}%")
print("Коэффициент Шарпа:", sharpe.get("sharperatio"))

Закрытых сделок: 93
Прибыльных сделок: 39
Убыточных сделок: 54
Максимальная просадка: 8.58%
Коэффициент Шарпа: 0.021920069819815688


#### Profit Factor, отношение суммарной прибыли выигрышных сделок к абсолютной сумме убытков проигрышных.

In [18]:
gross_profit = trades.won.pnl.total
gross_loss = abs(trades.lost.pnl.total)

profit_factor = (
    gross_profit / gross_loss
    if gross_loss > 0
    else float("inf")
)

print(f"Валовая прибыль выигрышных сделок: {gross_profit:,.2f}")
print(f"Суммарный убыток проигрышных сделок: {gross_loss:,.2f}")
print(f"Profit Factor: {profit_factor:.2f}")

Валовая прибыль выигрышных сделок: 624,575.66
Суммарный убыток проигрышных сделок: 387,837.86
Profit Factor: 1.61
